In [ ]:
import os
import json
import requests
from datetime import datetime
import pytz
from google_auth_oauthlib.flow import InstalledAppFlow

In [ ]:
all_media_items = []

In [ ]:
def get_access_token():
  SCOPES = ['https://www.googleapis.com/auth/photospicker.mediaitems.readonly']

  flow = InstalledAppFlow.from_client_secrets_file(
      'client_secret.json', SCOPES
  )

  credentials = flow.run_local_server(port=8080)

  print("Access token:", credentials.token)
  print("Scopes:", credentials.scopes)

  return credentials.token


access_token = get_access_token()

In [ ]:
def create_picker_session(access_token):
  headers = {
      'Authorization': f'Bearer {access_token}'
  }
  response = requests.post(
      'https://photospicker.googleapis.com/v1/sessions',
      headers=headers
  )

  if response.status_code == 200:
    return response.json()
  else:
    print("Error:", response.status_code, response.text)


session = create_picker_session(access_token)
if session:
  print("Picker session created successfully.")
  print("Session:", json.dumps(session, indent=2))

In [ ]:
def list_all_media_items(session_id, access_token, page_size=100):
  url = 'https://photospicker.googleapis.com/v1/mediaItems'
  headers = {
      'Authorization': f'Bearer {access_token}'
  }
  params = {
      'sessionId': session_id,
      'pageSize': page_size
  }
  all_media_items = []
  next_page_token = None

  while True:
    if next_page_token:
      params['pageToken'] = next_page_token
    response = requests.get(url, headers=headers, params=params)
    if response.status_code != 200:
      print("Error:", response.status_code, response.text)
      break

    data = response.json()
    items = data.get('mediaItems', [])
    all_media_items.extend(items)
    next_page_token = data.get('nextPageToken')
    if not next_page_token:
      break

  return all_media_items


session_id = session['id']
media_items = list_all_media_items(session_id, access_token)
print(f"Media items loaded in this session: {len(media_items)}")

all_media_items += media_items
print(f"All media items loaded: {len(all_media_items)}")

In [ ]:
all_media_items_ = {item['id']: item for item in all_media_items}
len(all_media_items_)

In [ ]:
for item in all_media_items_.values():
  media_file = item['mediaFile']
  filename = media_file.get('filename', 'Unknown')

  create_time = item['createTime']
  create_time = datetime.strptime(create_time, "%Y-%m-%dT%H:%M:%S.%fZ")
  create_time = create_time.replace(tzinfo=pytz.UTC)
  create_time = create_time.astimezone(pytz.timezone("Asia/Kolkata"))
  create_time = create_time.strftime("%B %d, %Y %I:%M %p")

  media_type = item['type']
  width = media_file.get('mediaFileMetadata', {}).get('width', 'Unknown')
  height = media_file.get('mediaFileMetadata', {}).get('height', 'Unknown')
  print(f"Filename: {filename}, Create Time: {create_time}, Type: {media_type}, Dimensions: {width}x{height}")

In [ ]:
screenshots_dir = r"C:\Users\k26ra\Pictures\Screenshots"
if os.path.exists(screenshots_dir):
  screenshots = os.listdir(screenshots_dir)
  print(f"Total screenshots found: {len(screenshots)}")
  for screenshot in screenshots:
    print(screenshot)